In [ ]:
# Install required libraries
# !pip install -q --upgrade openai numpy


## Tutorial: How Chunking Impacts RAG Performance
We'll compare two strategies on a small doc: fixed-size vs semantic (sentence) chunks.
Embeddings are generated through OpenRouter.


### 1) Configure OpenRouter API Key
Enter your OpenRouter API key when prompted.


In [ ]:
from getpass import getpass
from openai import OpenAI
import numpy as np

OPENROUTER_API_KEY = getpass("Enter your OpenRouter API key: ")

client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
)

EMBED_MODEL = "openai/text-embedding-3-small"

def embed(text: str):
    r = client.embeddings.create(model=EMBED_MODEL, input=text)
    return np.array(r.data[0].embedding, dtype=np.float32)

print("OpenRouter API key loaded:", bool(OPENROUTER_API_KEY))


### 2) Sample document
A short paragraph we'll chunk in two different ways.


In [ ]:
text = (
    "RAG combines retrieval with generation to improve factuality. "
    "Chunking documents helps retrieval find relevant pieces. "
    "Fixed-size chunks are simple but may split sentences. "
    "Sentence-based chunks keep semantics but can vary in length."
)
len(text)


### 3) Fixed-size chunker (with overlap)
Small function to split by characters, keeping overlap to preserve context.


In [ ]:
def fixed_chunk(text: str, size: int = 60, overlap: int = 10):
    chunks = []
    start = 0
    while start < len(text):
        end = min(len(text), start + size)
        chunks.append(text[start:end])
        if end == len(text):
            break
        start = end - overlap
    return chunks

fixed = fixed_chunk(text, size=60, overlap=10)
fixed


### 4) Sentence-based chunker
Split on `. ` to keep sentences intact (simple heuristic).


In [ ]:
def sentence_chunks(text: str):
    pieces = [p.strip() for p in text.split(". ") if p.strip()]
    # re-add period lost by split for all but last if needed
    chunks = [p if p.endswith(".") else p + "." for p in pieces]
    return chunks

sent_based = sentence_chunks(text)
sent_based


### 5) Embed both strategies
We'll embed all chunks from each strategy.


In [ ]:
fixed_embeds = [embed(c) for c in fixed]
sent_embeds = [embed(c) for c in sent_based]
len(fixed_embeds), len(sent_embeds)


### 6) Retrieval simulation
Embed a query and compute cosine similarity against each strategy's chunks.


In [ ]:
def cosine(a, b):
    na = np.linalg.norm(a) + 1e-10
    nb = np.linalg.norm(b) + 1e-10
    return float(np.dot(a, b) / (na * nb))

query = "How does chunking help RAG?"
qv = embed(query)

fixed_scores = [cosine(qv, v) for v in fixed_embeds]
sent_scores = [cosine(qv, v) for v in sent_embeds]

best_fixed = fixed[np.argmax(fixed_scores)]
best_sent = sent_based[np.argmax(sent_scores)]

best_fixed, best_sent


https://medium.com/@anuragmishra_27746/five-levels-of-chunking-strategies-in-rag-notes-from-gregs-video-7b735895694d